In [58]:
#pip install torch torchvision opencv-python

#TODO
#Split training data into 90-10 train, val(small validation set should speed up training)
#Implement Freezing early weights
#Do proper eval

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
from os.path import isfile, isdir
from os import makedirs

from datetime import datetime           #File timstamp
import time                             #T+121239
from PIL import Image                   #Needed to load random image
import cv2 as cv                        #Show images and stuff(Can also be done with PIL
import matplotlib.pyplot as plt         #Show images for colab
import numpy as np                      #Honestly this should just be a core library

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
Timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

In [7]:
class LeNet(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=9, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=9, out_channels=27, kernel_size=5)
        self.fc_1 = nn.Linear(27 * 25 * 25, 120)
        self.fc_2 = nn.Linear(120, 84)
        self.fc_3 = nn.Linear(84, output_dim)

    def forward(self, x):
        x = self.conv1(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = F.relu(x)
        x = x.view(x.shape[0], -1)
        x = self.fc_1(x)
        x = F.relu(x)
        x = self.fc_2(x)
        x = F.relu(x)
        x = self.fc_3(x)
        return x

In [8]:
#Pretrained model weight None>Random, "DEFAULT">torch MobileNetV2, "IMAGENET1K_V1">original MobileNetV2, or "FILENAME" for custom
PTWeights = "DEFAULT"
PTFreeze = False                             #Freeze Non-Output layer
PTOutputs = 7                             #Alter Output size if different from loaded network (WILL wipe weights for this)
TrainingRate = 0.01
TrainingEpochs = 100
EarlyStop = 10

PreProcess = v2.Compose([                   #Preprocessing steps to get dataset to appropriate format
    #v2.Grayscale(1),
    #v2.Resize(224),                         #Resizes to Xpx short side
    v2.Resize(None, max_size=100),          #Resizes to Xpx long side
    v2.Pad(20),
    v2.CenterCrop(100),                     #Crops excess on long side
    v2.ToImage(),                           #Makes it an image???
    v2.ToDtype(torch.float32, scale = True), #Changes datatype to float and value range from 0-255 to 0-1
    #v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),    #Normalizes image because apparently the (pretrained)model likes that
])

model = LeNet(PTOutputs)
if PTWeights and isfile(PTWeights): model.load_state_dict(torch.load(PTWeights, weights_only=True))
#if PTOutputs != model.classifier[1].out_features: model.classifier[1] = nn.Linear(model.classifier[1].in_features, PTOutputs)
if PTFreeze:
    for param in model.parameters():
        param.requires_grad = False
    model.classifier[-1].weight.requires_grad = True
    model.classifier[-1].bias.requires_grad = True

In [9]:
AnnotationLevel = "family"
AnnotationFile = "./fgvc-aircraft-2013b/data/families.txt"

In [12]:
import json

import os, fnmatch
def find(pattern, path):
    result = []
    for root, dirs, files in os.walk(path):
        for name in files:
            if fnmatch.fnmatch(name, pattern):
                result.append(os.path.join(root, name))
    return result



In [20]:
######Custom data loading
from os import listdir

Folder = "../OURdata/"
Labels = {
    'dirty': 0,
    'damaged': 1,
    'vandalized': 2,
    'faded': 3,
    'obscured': 4
}

class OURdata(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Arguments:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.data = []
        self.labels = []

        with open(Folder+'OURLabels.json', 'r',encoding="utf-8") as f:
            allJJJ = json.load(f)
        print(allJJJ[0])
        for jasonOne in allJJJ:
            imName = jasonOne['file_upload'][9:]
            extOne = imName.rfind('.')
            extTwo = imName.rfind('_')
            files = find(imName[:extOne] + '_*', root_dir)
            print(imName)
            print(files)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        sample = self.data[idx]
        label = self.labels[idx]
        
        if self.transform:
            sample = self.transform(sample)

        return sample, label
FullData = OURdata(Folder, PreProcess)
TrainData, TestData = torch.utils.data.random_split(FullData, [0.8, 0.2])

{'id': 1, 'annotations': [{'id': 15, 'completed_by': 1, 'result': [{'value': {'choices': ['dirty']}, 'id': 'mmWbEU_uU9', 'from_name': 'choice', 'to_name': 'image', 'type': 'choices', 'origin': 'manual'}], 'was_cancelled': False, 'ground_truth': False, 'created_at': '2026-11-18T11:18:07.608082Z', 'updated_at': '2026-11-18T11:18:07.608082Z', 'draft_created_at': '2026-11-18T11:17:50.507913Z', 'lead_time': 91.916, 'prediction': {}, 'result_count': 1, 'unique_id': 'e1d100dc-6283-4b27-ac9a-af2c8817bfb6', 'import_id': None, 'last_action': None, 'bulk_created': False, 'task': 1, 'project': 6, 'updated_by': 1, 'parent_prediction': None, 'parent_annotation': None, 'last_created_by': None}], 'file_upload': 'e0e4431a-30_zone.jpeg', 'drafts': [], 'predictions': [], 'data': {'image': '/data/upload/6/e0e4431a-30_zone.jpeg'}, 'meta': {}, 'created_at': '2026-11-18T11:15:32.983346Z', 'updated_at': '2026-11-18T11:18:07.740847Z', 'allow_skip': True, 'inner_id': 1, 'total_annotations': 1, 'cancelled_annota

C:\Users\ditte\AppData\Local\Temp\ipykernel_17040\658668948.py:53: UserWarning: Length of split at index 0 is 0. This might result in an empty dataset.
  TrainData, TestData = torch.utils.data.random_split(FullData, [0.8, 0.2])
C:\Users\ditte\AppData\Local\Temp\ipykernel_17040\658668948.py:53: UserWarning: Length of split at index 1 is 0. This might result in an empty dataset.
  TrainData, TestData = torch.utils.data.random_split(FullData, [0.8, 0.2])


In [16]:
print(FullData[0][0].shape)
display(v2.ToPILImage()(FullData[0][0]))

IndexError: list index out of range

In [65]:
#TrainData = datasets.FGVCAircraft(root="./", split="train", annotation_level=AnnotationLevel, download=True, transform=PreProcess)
#TestData = datasets.FGVCAircraft(root="./", split="test", annotation_level=AnnotationLevel, download=True, transform=PreProcess)
TrainDataLoader = torch.utils.data.DataLoader(TrainData, batch_size=80, shuffle=True, pin_memory=True)#, num_workers=args.nThreads
TestDataLoader = torch.utils.data.DataLoader(TestData, batch_size=80, shuffle=True, pin_memory=True)#, num_workers=args.nThreads

In [66]:
annotations = ["You are at peace", "Right", "Left", "RotRight", "RotLeft", "SoftDrop", "HardDrop"]
#with open(AnnotationFile, 'rb') as file:
#    for line in file:
#        annotations.append(line.decode("utf-8").strip())

In [67]:
criterion = nn.CrossEntropyLoss()       #Multiclass classification
optimizer = torch.optim.Adam(model.parameters(), lr=TrainingRate)
def TrainEpoch():                       #Trains once on all training data
    accumulatedLoss = 0.0
    #Train on the whole dataset but tragically my GPU doesnt fit the whole dataset
    #(which is weird because its only a few GB) so it has to be in batches
    for BatchID, Batch in enumerate(TrainDataLoader):
        BatchIm, BatchLabel = Batch
        BatchIm, BatchLabel = BatchIm.to(device), BatchLabel.to(device)     #Put stuff on GPU(if cuda is available)

        optimizer.zero_grad()                   #Reset gradients
        output = model(BatchIm)                 #Make predictions
        loss = criterion(output, BatchLabel)    #How wrong are we
        loss.backward()                         #Change model weights based on how wrong we are
        optimizer.step()
        accumulatedLoss += loss.item()
    return accumulatedLoss / TrainDataLoader.batch_size

In [68]:
def TrainEpochs(Epochs):
    
    if not isdir(f"./{model.__class__.__name__}/"): makedirs(f"./{model.__class__.__name__}/")
    model.to(device)
    BestTest = np.inf
    NoImprovement = 0
    tStart = time.time()
    for Epoch in range(Epochs):
        model.train(True)
        print(f"Train T+{time.time() - tStart:>5.0f}")
        LossTrain = TrainEpoch()
    
        LossTest = 0.0
        model.eval()
        with torch.no_grad():
            print(f"Eval  T+{time.time() - tStart:>5.0f}")
            for BatchID, Batch in enumerate(TestDataLoader):
                BatchIm, BatchLabel = Batch
                BatchIm, BatchLabel = BatchIm.to(device), BatchLabel.to(device)     #Put stuff on GPU(if cuda is available)
                output = model(BatchIm)                                             #Make predictions
                LossTest += criterion(output, BatchLabel)                           #How wrong are we
    
            LossTest = LossTest / TestDataLoader.batch_size
    
        if LossTest < BestTest:
            BestTest, NoImprovement = LossTest, 0
            torch.save(model.state_dict(), f"./{model.__class__.__name__}/{Timestamp}_best")
        else:
            NoImprovement += 1
            if NoImprovement > EarlyStop:
                print(f"Training does not continue until morale improves")
                torch.save(model.state_dict(), f"./{model.__class__.__name__}/{Timestamp}_final")
                break
        print(f"Epoch [{Epoch+1:>3}/{Epochs:>3}] T+{time.time() - tStart:>5.0f}, Training Loss: {LossTrain:>6.3f}, Test Loss: {LossTest:>6.3f},  Worse then best [{NoImprovement:>2}/{EarlyStop:>2}]")

In [69]:
TrainEpochs(TrainingEpochs)

Train T+    0


RuntimeError: addmm: mat1 and mat2 shapes cannot be multiplied (80x13068 and 16875x120)

In [ ]:
numberID = 207
testImage, Label = TestData[numberID]
print(f"It is a ({Label}){annotations[Label]}", testImage.shape)
testTensor = PreProcess(testImage).unsqueeze(0).to(device)  #Put the image through the preprocessing, then unsqueeze to add an extra dimension(image is 3D this model wants 4D

result = model(testTensor)
probabilities = F.softmax(result, dim=1).squeeze()
topProb, topClass = torch.topk(probabilities, 5)
for i in range(len(topProb)):
    print(f"I am {topProb[i].item():.2f} percent sure this is a {"annotations[topClass[i].item()]"} Class:{topClass[i].item()}")

It is a (6)HardDrop torch.Size([3, 28, 28])
I am 0.25 percent sure this is a annotations[topClass[i].item()] Class:5
I am 0.15 percent sure this is a annotations[topClass[i].item()] Class:4
I am 0.13 percent sure this is a annotations[topClass[i].item()] Class:6
I am 0.13 percent sure this is a annotations[topClass[i].item()] Class:0
I am 0.13 percent sure this is a annotations[topClass[i].item()] Class:3


In [ ]:
#from google.colab.patches import cv2_imshow
#testImage, Label = TestData[numberID]
#testImage = testImage.permute(1, 2, 0).numpy() * 200
#cv2_imshow(testImage)
#print(Label)

In [ ]:
def count_parameters(model):
    print(f"{"Layer name":^30}|{"Parameters":>10}|{"Trainable":>10}")
    total_params, train_params = 0, 0
    for name, parameter in model.named_parameters():
        #if not parameter.requires_grad: continue
        params = parameter.numel()
        print(f"{name:<30}|{params:>10}|{params*parameter.requires_grad:>10}")
        total_params += params
        if parameter.requires_grad: train_params += params
    print(f"{"Total parameters":<30}|{total_params:>10}|{train_params:>10}")
count_parameters(model)

          Layer name          |Parameters| Trainable
conv1.weight                  |       675|       675
conv1.bias                    |         9|         9
conv2.weight                  |      6075|      6075
conv2.bias                    |        27|        27
fc_1.weight                   |     51840|     51840
fc_1.bias                     |       120|       120
fc_2.weight                   |     10080|     10080
fc_2.bias                     |        84|        84
fc_3.weight                   |       588|       588
fc_3.bias                     |         7|         7
Total parameters              |     69505|     69505


Some things i learned:
freezing all parameters except classifier layer reduces GPU usage during training a *lot*, so we might be able to train a model on UCloud decently fast if we just fine tune? It does not actually train faster on my computer but thats because CPU is the limit for me

possibly because of backpropagation not needing to calculate so many gradients, possibly due to just not tracking gradients
fully trimmed model i tested also replaced classifier layer with one that only had 100 outputs instead of 1000
since MobileNetV2 is trained on a dataset with 1000 classes and we only have a bit under 100

this means it only trains 128100 parameters instead of 3504872, and is what is called "fine tuning"(i think) since you leave almost the whole model intact and just retrain the output layers

the fully trimmed down model used only about 2GB VRAM and barely any work on GPU

In [ ]:
print(model)

LeNet(
  (conv1): Conv2d(3, 9, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(9, 27, kernel_size=(5, 5), stride=(1, 1))
  (fc_1): Linear(in_features=432, out_features=120, bias=True)
  (fc_2): Linear(in_features=120, out_features=84, bias=True)
  (fc_3): Linear(in_features=84, out_features=7, bias=True)
)
